# Early Sepsis Alert System (6-Hour Ahead Prediction)

Hackathon Prototype

This project develops a clinically realistic early warning system
for sepsis using ICU time-series data.

Focus: Reduce alert fatigue while maintaining early detection capability.

## Load Data

In [11]:
import pandas as pd

baseline_df = pd.read_csv(
    "/content/drive/MyDrive/physionet/baseline_dataset.psv",
    sep="|"
)

print("Shape:", baseline_df.shape)
print("Patients:", baseline_df["PatientID"].nunique())
print("Positive rate (%):", baseline_df["FutureSepsis"].mean() * 100)

Shape: (790215, 14)
Patients: 20336
Positive rate (%): 1.1255164733648437


## Patient-Wise Train/Test Split

In [12]:
from sklearn.model_selection import train_test_split

patients = baseline_df['PatientID'].unique()

train_patients, test_patients = train_test_split(
    patients,
    test_size=0.2,
    random_state=42
)

train_df = baseline_df[baseline_df['PatientID'].isin(train_patients)].copy()
test_df  = baseline_df[baseline_df['PatientID'].isin(test_patients)].copy()

print("Train patients:", train_df['PatientID'].nunique())
print("Test patients:", test_df['PatientID'].nunique())

Train patients: 16268
Test patients: 4068


## Missing Handling

In [13]:
lab_cols = ['Lactate','WBC','Creatinine','Platelets']
vitals = ['HR','O2Sat','SBP','MAP','Resp','Temp']

# Missing indicators
for col in lab_cols:
    train_df[col + '_missing'] = train_df[col].isnull().astype(int)
    test_df[col + '_missing']  = test_df[col].isnull().astype(int)

# Forward fill vitals per patient
train_df[vitals] = train_df.groupby('PatientID')[vitals].ffill()
test_df[vitals]  = test_df.groupby('PatientID')[vitals].ffill()

# Median fill (train statistics only)
vital_medians = train_df[vitals].median()
train_df[vitals] = train_df[vitals].fillna(vital_medians)
test_df[vitals]  = test_df[vitals].fillna(vital_medians)

## **Feature Engineering** (Final Version - v6)

In [14]:
# Delta6
for col in vitals:
    train_df[col + '_delta6'] = (
        train_df.groupby('PatientID')[col].shift(0) -
        train_df.groupby('PatientID')[col].shift(6)
    ).fillna(0)

    test_df[col + '_delta6'] = (
        test_df.groupby('PatientID')[col].shift(0) -
        test_df.groupby('PatientID')[col].shift(6)
    ).fillna(0)

# Delta1
for col in vitals:
    train_df[col + '_delta1'] = (
        train_df.groupby('PatientID')[col].shift(0) -
        train_df.groupby('PatientID')[col].shift(1)
    ).fillna(0)

    test_df[col + '_delta1'] = (
        test_df.groupby('PatientID')[col].shift(0) -
        test_df.groupby('PatientID')[col].shift(1)
    ).fillna(0)

# Volatility (6-hour std)
for col in vitals:
    train_df[col + '_roll6_std'] = (
        train_df.groupby('PatientID')[col]
        .rolling(window=6, min_periods=1)
        .std()
        .reset_index(level=0, drop=True)
    ).fillna(0)

    test_df[col + '_roll6_std'] = (
        test_df.groupby('PatientID')[col]
        .rolling(window=6, min_periods=1)
        .std()
        .reset_index(level=0, drop=True)
    ).fillna(0)

## **Train Final Model** (HistGradientBoosting)

In [15]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

X_train = train_df.drop(columns=['FutureSepsis','PatientID'])
y_train = train_df['FutureSepsis']

X_test  = test_df.drop(columns=['FutureSepsis','PatientID'])
y_test  = test_df['FutureSepsis']

hgb_model = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.1,
    max_iter=200,
    random_state=42
)

hgb_model.fit(X_train, y_train)

y_pred_proba = hgb_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("PR-AUC:", average_precision_score(y_test, y_pred_proba))

ROC-AUC: 0.7480950680155708
PR-AUC: 0.03999323589569209


## **Alert System Design** (Final Deployment Logic)

In [16]:
threshold = 0.04

test_df_alert = test_df.copy()
test_df_alert["prob"] = y_pred_proba

test_df_alert = test_df_alert.sort_values(["PatientID", "ICULOS"])

# High risk flag
test_df_alert["high_risk"] = (test_df_alert["prob"] >= threshold).astype(int)

# 2-hour persistence rule
test_df_alert["high_risk_prev"] = (
    test_df_alert.groupby("PatientID")["high_risk"]
    .shift(1)
    .fillna(0)
)

test_df_alert["alert"] = (
    (test_df_alert["high_risk"] == 1) &
    (test_df_alert["high_risk_prev"] == 1)
).astype(int)

# Patient-level evaluation
patient_alerts = test_df_alert.groupby("PatientID").agg({
    "alert": "max",
    "FutureSepsis": "max"
}).reset_index()

TP = ((patient_alerts["alert"] == 1) & (patient_alerts["FutureSepsis"] == 1)).sum()
FP = ((patient_alerts["alert"] == 1) & (patient_alerts["FutureSepsis"] == 0)).sum()
FN = ((patient_alerts["alert"] == 0) & (patient_alerts["FutureSepsis"] == 1)).sum()

precision = TP / (TP + FP)
recall = TP / (TP + FN)

print("Final Patient-Level Precision:", precision)
print("Final Patient-Level Recall:", recall)

Final Patient-Level Precision: 0.49477351916376305
Final Patient-Level Recall: 0.459546925566343


# **Conclusion (Markdown)**

## Final System Summary

We developed a clinically realistic early warning model:

• HistGradientBoosting on engineered temporal features  
• 6-hour ahead prediction target  
• Patient-wise train-test split  
• 2-hour consecutive persistence alert rule  

### Final Deployment Performance (Patient-Level)

Precision ≈ 49%  
Recall ≈ 46%  

By introducing persistence logic, we significantly reduced false alarms
while maintaining strong early detection capability.

This makes the system more suitable for real ICU deployment.